this here is the data pipeline for turning the data into tuples!

In [ ]:
#load packages
import numpy as np
import mne
import pandas as pd
from scipy.signal import coherence
from scipy.signal import welch

In [ ]:
# loading data from:https://physionet.org/content/eegmat/1.0.0/
# dataset has a low-pass filter at 30Hz so gamma (30+Hz) has minimal signal
# _1 files = baseline EEG, _2 files = EEG during mental arithmetic task
# load files in subject order, trim each one before combining
files = [
    ('Subject00_1.edf', 0, 'baseline'),
    ('Subject00_2.edf', 0, 'task'),
    ('Subject01_1.edf', 1, 'baseline'),
    ('Subject01_2.edf', 1, 'task'),
    ('Subject02_1.edf', 2, 'baseline'),
    ('Subject02_2.edf', 2, 'task'),
]
trimmed = []

for f, subject_id, session in files:
    raw = mne.io.read_raw_edf(f, preload=True, encoding='latin1')
    temp = raw.to_data_frame()
    temp = temp.drop(columns=['ECG ECG'])
    
    # trim flat padding off the end of each file
    eeg_data = temp[[col for col in temp.columns if col.startswith('EEG')]]
    changing = (eeg_data.diff() != 0).any(axis=1)
    last_valid = changing[::-1].idxmax()
    temp = temp.iloc[:last_valid + 1]
    
    # tag each row with subject and session so teammates know the source
    temp['subject_id'] = subject_id
    temp['session'] = session
    
    print(f'{f}: {len(temp)} valid rows')
    trimmed.append(temp)

df = pd.concat(trimmed, ignore_index=True)
print(f'Total rows: {len(df)}')
df

In [3]:
#make full second sized windows for proper frequency resolution
fs=500
window_size=500
n_windows=len(df)//window_size

#get all channels as long as they are eeg channels
channels = []
for col in df.columns:
    if col.startswith('EEG'):
        channels.append(col)

print(f"n_windows: {n_windows}")
print(f"expected coherence rows: {n_windows * 190}")
print(f"channels: {channels}")

n_windows: 726
expected coherence rows: 137940
channels: ['EEG Fp1', 'EEG Fp2', 'EEG F3', 'EEG F4', 'EEG F7', 'EEG F8', 'EEG T3', 'EEG T4', 'EEG C3', 'EEG C4', 'EEG T5', 'EEG T6', 'EEG P3', 'EEG P4', 'EEG O1', 'EEG O2', 'EEG Fz', 'EEG Cz', 'EEG Pz', 'EEG A2-A1']


In [ ]:
# band definitions for coherence
# delta, theta, alpha, beta have strong signal
# gamma has minimal signal (data is low-pass filtered at 30Hz) but included for completeness
BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta': (13, 30),
    'gamma': (30, 45)
}

# window loop to find band-specific coherence of pairs per window
# no pre-filtering needed: scipy.signal.coherence already returns per-frequency values
# we just mask the output by band range (same technique as band power)
output_list = []
for w in range(n_windows):
    print(f"window #: {w} of {n_windows}")
    start=w*window_size
    end=start+window_size
    window_data=df.iloc[start:end]
    
    # grab subject_id and session from the middle of the window
    mid = start + window_size // 2
    subj = df.iloc[mid]['subject_id']
    sess = df.iloc[mid]['session']

    for i in range(len(channels)):
        for j in range(i+1, len(channels)):
            channel1 = channels[i]
            channel2 = channels[j]
            freqs, coh = coherence(window_data[channel1].values, window_data[channel2].values, fs=500.0, nperseg=256)
            
            # extract coherence per band using frequency masks
            row = {'epoch_id': w, 'subject_id': subj, 'session': sess, 'channel_i': channel1, 'channel_j': channel2}
            for band, (lo, hi) in BANDS.items():
                mask = (freqs >= lo) & (freqs < hi)
                row[band + '_coherence'] = coh[mask].mean()
            
            output_list.append(row)

output_df = pd.DataFrame(output_list)

# clean up bad values from dying signal at file boundaries
output_df = output_df.dropna()
output_df = output_df[output_df['delta_coherence'] < 1.0]
output_df = output_df[output_df['theta_coherence'] < 1.0]
output_df = output_df[output_df['alpha_coherence'] < 1.0]
output_df = output_df[output_df['beta_coherence'] < 1.0]
print(f"Total coherence rows: {len(output_df)}")
output_df

In [ ]:
output_df.to_csv('TDG_trial_run_data.csv')

In [ ]:
# band power per channel per window
# delta, theta, alpha, beta have strong signal
# gamma has minimal signal (data is low-pass filtered at 30Hz) but included for completeness
BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta': (13, 30),
    'gamma': (30, 45)
}

band_list = []
for w in range(n_windows):
    print(f"Processing band power window {w} of {n_windows}")
    start=w*window_size
    end=start+window_size
    window_data=df.iloc[start:end]
    
    # grab subject_id and session from the middle of the window
    mid = start + window_size // 2
    subj = df.iloc[mid]['subject_id']
    sess = df.iloc[mid]['session']
    
    for channel in channels:
        segment=window_data[channel].values
        freqs, psd=welch(segment, fs=500.0, nperseg=256)
        
        powers = {}
        for band, (lo, hi) in BANDS.items():
            mask = (freqs >= lo) & (freqs < hi)
            powers[band] = np.sum(psd[mask])
        
        total = sum(powers.values())
        for band in powers:
            powers[band] = powers[band] / total
        
        dominant = max(powers, key=powers.get)
        
        row = {
            'epoch_id': w,
            'subject_id': subj,
            'session': sess,
            'channel': channel,
            'delta': powers['delta'],
            'theta': powers['theta'],
            'alpha': powers['alpha'],
            'beta': powers['beta'],
            'gamma': powers['gamma'],
            'dominant_band': dominant
        }
        band_list.append(row)

band_df = pd.DataFrame(band_list)

# clean up bad values from dying signal at file boundaries
band_df = band_df.dropna()
print(f"Total band power rows: {len(band_df)}")
band_df

In [ ]:
band_df.to_csv('TDG_band_power_data.csv')